# 自動未入帳清單產生器


In [ ]:
import pandas as pd
import json
import os
import warnings
import itertools
# import google.generativeai as genai
from IPython.display import display, HTML
import win32com.client as win32
import os
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_columns', None)

import openai
# 移除 import google.generativeai as genai 那行
api_key = os.environ.get("OPENAI_API_KEY", "")  # 請設定環境變數 OPENAI_API_KEY
client = openai.OpenAI(api_key=api_key)

# 2. 自動掃描資料夾，找對應檔案（支援 xls / xlsm / xlsx）
data_dir = r'C:\Users\user\Desktop\專案\會計對帳\資料'
bank_path = None
acc_path = None
for fname in os.listdir(data_dir):
    fpath = os.path.join(data_dir, fname)
    if fname.startswith('~$'):
        continue
    if '銀行對帳單' in fname and fname.endswith(('.xlsx', '.xls', '.xlsm')):
        bank_path = fpath
    elif '帳務查詢' in fname and fname.endswith(('.xlsx', '.xls', '.xlsm')):
        acc_path = fpath

assert bank_path, '找不到包含「銀行對帳單」的檔案'
assert acc_path,  '找不到包含「帳務查詢」的檔案'

# 3. 若非 xlsx，轉換後覆蓋路徑變數
def convert_to_xlsx(src_path):
    if src_path.endswith('.xlsx'):
        # 先嘗試 openpyxl 能不能開（避免偽裝的 xls）
        try:
            from openpyxl import load_workbook
            load_workbook(src_path, read_only=True).close()
            return src_path
        except Exception:
            pass  # 偽裝的 xls，繼續往下轉

    import win32com.client, pythoncom
    pythoncom.CoInitialize()
    dst_path = src_path.rsplit('.', 1)[0] + '.xlsx'
    excel = win32com.client.Dispatch('Excel.Application')
    excel.Visible = False
    excel.DisplayAlerts = False
    try:
        wb = excel.Workbooks.Open(src_path)
        wb.SaveAs(dst_path, FileFormat=51)  # 51 = xlsx
        wb.Close(False)
        print(f'🔄 已轉換（保留格式）：{os.path.basename(dst_path)}')
    finally:
        excel.Quit()
        pythoncom.CoUninitialize()
    return dst_path


bank_path = convert_to_xlsx(bank_path)
acc_path  = convert_to_xlsx(acc_path)

print(f'✅ bank_path = {bank_path}')
print(f'✅ acc_path  = {acc_path}')


base_name = '未入帳清單_'
output_html = os.path.join(data_dir, base_name + '.html')
output_xlsx = os.path.join(data_dir, base_name + '.xlsx')
counter = 1
while os.path.exists(output_xlsx):
    output_xlsx = os.path.join(data_dir, f'{base_name}_{counter}.xlsx')
    counter += 1


## 讀取與數據分流

In [ ]:
def load_and_preprocess(target_date):
    # --- 銀行端 ---
    df_bank = pd.read_excel(bank_path, skiprows=5)
    df_bank = df_bank.dropna(subset=['交易日期'])
    df_bank['存入金額'] = pd.to_numeric(df_bank['存入金額'].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(0)
    df_bank['支出金額'] = pd.to_numeric(df_bank['支出金額'].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(0)
    
    # 銀行端用整個月
    target_month = target_date[:7]  # '2026/04'
    df_bank_all = df_bank[df_bank['交易日期'].astype(str).str.contains(target_month.replace('/', '/0') if '/0' not in target_month else target_month)].copy()
    # 更保險的寫法：
    df_bank_all = df_bank.copy()
    df_bank_all['_date_str'] = pd.to_datetime(df_bank_all['交易日期'], errors='coerce').dt.strftime('%Y/%m')
    df_bank_all = df_bank_all[df_bank_all['_date_str'] == target_month].drop(columns=['_date_str'])
    df_bank_all['Bank_ID'] = df_bank_all.index.astype(str)
    
    bank_in  = df_bank_all[df_bank_all['存入金額'] > 0].copy()
    bank_out = df_bank_all[df_bank_all['支出金額'] > 0].copy()
    
    # --- 會計端（合併 未過賬 + 已過賬）---
    required_acc_columns = [
        '憑證創建人', '憑證編號', '憑證行編號', '憑證類型', '業務參考', 
        '會計期間', '業務日期', '科目代碼', '描述.1', '業務貨幣代碼', 
        '業務金額', '本位幣金額', '第二本位幣/報表金額', '借方/貸方', 
        '商部別 分析代碼', '定存單號/銀行與帳號 分析代碼', '對象別 分析代碼'
    ]
    
    all_sheets = pd.read_excel(acc_path, sheet_name=None)
    df_acc_list = []
    for sheet_name, df_sheet in all_sheets.items():
        actual_cols = [c for c in df_sheet.columns if c in required_acc_columns or c == '描述.1' or c == '業務金額' or c == '業務日期']
        if '業務金額' not in df_sheet.columns or '業務日期' not in df_sheet.columns:
            continue  # 跳過非資料 sheet
        df_sheet = df_sheet[actual_cols].copy()
        df_sheet['來源Sheet'] = sheet_name
        df_acc_list.append(df_sheet)
        print(f'  ✅ 讀入 [{sheet_name}]：{len(df_sheet)} 筆')
    
    df_acc = pd.concat(df_acc_list, ignore_index=True)
    def parse_amount(s):
        s = str(s).strip().replace(',', '')
        if s.startswith('(') and s.endswith(')'):
            s = '-' + s[1:-1]   # (19705) → -19705
        return pd.to_numeric(s, errors='coerce')
    df_acc['業務金額'] = df_acc['業務金額'].apply(parse_amount).fillna(0)
    
    try:
        df_acc['業務日期_格式化'] = pd.to_datetime(df_acc['業務日期'], errors='coerce').dt.strftime('%Y/%m/%d')
        if df_acc['業務日期_格式化'].isna().all():
            raise ValueError()
    except:
        df_acc['業務日期_格式化'] = pd.to_datetime(
            pd.to_numeric(df_acc['業務日期'], errors='coerce'),
            origin='1899-12-30', unit='D'
        ).dt.strftime('%Y/%m/%d')
    
    # 會計端用整個月過濾
    target_month = target_date[:7]  # '2026/04'
    df_acc_all = df_acc[df_acc['業務日期_格式化'].str.startswith(target_month, na=False)].copy()
    df_acc_all['Acc_ID'] = df_acc_all.index.astype(str)
    
    acc_in  = df_acc_all[df_acc_all['業務金額'] > 0].copy()
    acc_out = df_acc_all[df_acc_all['業務金額'] < 0].copy()
    
    return bank_in, bank_out, acc_in, acc_out

target_date = '2026/04'
bank_in, bank_out, acc_in, acc_out = load_and_preprocess(target_date)
print(f"✅ 數據載入：[收入組] 銀行 {len(bank_in)} 筆 / 會計 {len(acc_in)} 筆 | [支出組] 銀行 {len(bank_out)} 筆 / 會計 {len(acc_out)} 筆")


In [ ]:
import re

def extract_keywords(text):
    """從附言/描述.1 抽出有意義的關鍵字"""
    text = str(text).strip()
    if not text or text.lower() == 'nan':
        return []
    # 英數字≥3碼 或 中文≥2字
    tokens = re.findall(r'[A-Za-z0-9]{3,}|[\u4e00-\u9fff]{2,}', text)
    return [t.upper() for t in tokens]

def keyword_match(bank_memo, acc_desc1):
    """
    銀行附言 vs 會計描述.1 關鍵字匹配
    回傳 (是否匹配, 匹配到的關鍵字)
    """
    bank_tokens = extract_keywords(bank_memo)
    acc_text = str(acc_desc1).upper()
    for token in bank_tokens:
        if token in acc_text:
            return True, token
    # 反向：acc 關鍵字在 bank 附言裡
    bank_text = str(bank_memo).upper()
    for token in extract_keywords(acc_desc1):
        if len(token) >= 4 and token in bank_text:
            return True, token
    return False, ''


## 對帳

In [ ]:
def check_memo_match(bank_memo, acc_desc_extra):
    b_text = str(bank_memo).strip()
    a_text = str(acc_desc_extra).strip()
    if not b_text or b_text.lower() in ['nan']: return True 
    if a_text and a_text.lower() not in ['nan']:
        if (b_text in a_text) or (a_text in b_text) or len(set(b_text) & set(a_text)) >= 2: return True
    return False

def llm_pick_best_candidate(bank_row, candidates_df, mode):
    # ✅ 新增：若所有候選會計帳描述含「銀行存款－台北富邦銀行」→ LLM 無法分辨，直接返回 None
    if '描述.1' in candidates_df.columns:
        if candidates_df['描述.1'].astype(str).str.contains('銀行存款－台北富邦銀行', na=False).all():
            print(f"      ⏭️ 所有候選描述均為銀行存款，LLM 無法分辨，留待批次核銷")
            return None
    
    prompt = f"""你是專業會計。有一筆銀行{('收款' if mode=='Income' else '付款')}，與其金額相同的會計帳有 {len(candidates_df)} 筆候選紀錄。
請根據銀行的「附言」與會計帳的「備註描述(描述.1)」，挑出真正對應的「唯一 1 筆」會計帳。
【目標銀行紀錄】
 Bank_ID: {bank_row['Bank_ID']}, 附言: {bank_row['附言']}
【多筆會計候選人】
{candidates_df[['Acc_ID', '描述.1']].to_json(orient='records', force_ascii=False)}
請只回傳最符合的該筆 Acc_ID 字串。若都不吻合則回傳 None。"""
    try:
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0
        )
        ans = resp.choices[0].message.content.strip()
        if ans in candidates_df['Acc_ID'].values: return ans
    except: pass
    return None


def evaluate_combination_with_llm(bank_rows, acc_rows, match_type, mode):
    amt_col_b = '存入金額' if mode == 'Income' else '支出金額'
    
    # ✅ 新增：若會計描述含「銀行存款－台北富邦銀行」→ 描述無鑑別度，數學吻合直接通過
    if '描述.1' in acc_rows.columns:
        if acc_rows['描述.1'].astype(str).str.contains('銀行存款－台北富邦銀行', na=False).all():
            return True, "會計描述為銀行存款類，無需LLM，數學完全吻合自動通過"
    
    prompt = f"""你是專業會計。系統發現一個「金額完美對齊」的對帳組合 ({match_type}, 方向: {mode})。
請考察「銀行帳附言」與「會計帳描述.1」，判斷這組資料是否為同一筆業務的拆帳或合併入帳。
【銀行帳紀錄】
{bank_rows[['Bank_ID', '附言']].to_json(orient='records', force_ascii=False)}
【會計帳紀錄】
{acc_rows[['Acc_ID', '描述.1']].to_json(orient='records', force_ascii=False)}
請只回傳純 JSON: {{"is_valid": true/false, "reason": "..."}}"""
    try:
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
            response_format={'type': 'json_object'}
        )
        ans = json.loads(resp.choices[0].message.content)
        return ans.get('is_valid', False), ans.get('reason', '')
    except: return False, 'AI 解析失敗'

# ── 重新定義加入 N:1 邏輯的 reconcile_engine ──
def reconcile_engine(df_bank, df_acc, mode='Income', pending=None):
    if pending is None: pending = []
    label = "收入" if mode == "Income" else "支出"
    print(f"\n{'='*16} 🚀 [{label}] 開始對帳 {'='*16}")
    rem_b, rem_a = df_bank.copy(), df_acc.copy()
    history = []
    bank_amt_col = '存入金額' if mode == 'Income' else '支出金額'
    # ── Step 1: 1對1 ──
    print(f"\n⏳ [Step 1] 1對1 金額碰撞...")
    for b_idx, b_row in df_bank.iterrows():
        b_amt = b_row[bank_amt_col]
        candidates = rem_a[rem_a['業務金額'].abs() == b_amt]
        if len(candidates) == 1:
            a_row = candidates.iloc[0]
            km, kw = keyword_match(b_row.get('附言',''), a_row.get('描述.1',''))
            reason = f'金額唯一 + 關鍵字「{kw}」' if km else '金額唯一'
            print(f"  ✅ 1:1 Bank:{b_row['Bank_ID']}({b_amt:,.0f}) ↔ Acc:{a_row['Acc_ID']} [{reason}]")
            history.append({'Type': f'{label} 1對1', 'MatchReason': reason,
                            'Bank_Data': b_row.to_frame().T, 'Acc_Data': a_row.to_frame().T})
            rem_a = rem_a.drop(candidates.index[0])
            rem_b = rem_b.drop(b_idx)
        elif len(candidates) > 1:
            # 先關鍵字
            kw_winner = None
            for _, cand in candidates.iterrows():
                km, kw = keyword_match(b_row.get('附言',''), cand.get('描述.1',''))
                if km:
                    kw_winner = (cand, kw); break
            
            if kw_winner:
                a_sel, kw = kw_winner
                reason = f'金額相符 + 關鍵字「{kw}」'
                print(f"  ✅ 1:1(關鍵字) Bank:{b_row['Bank_ID']} ↔ Acc:{a_sel['Acc_ID']} [{reason}]")
                history.append({'Type': f'{label} 1對1', 'MatchReason': reason,
                                'Bank_Data': b_row.to_frame().T, 'Acc_Data': a_sel.to_frame().T})
                rem_a = rem_a[rem_a['Acc_ID'] != a_sel['Acc_ID']]
                rem_b = rem_b.drop(b_idx)
            else:
                # 再 LLM
                print(f"  ⚠️ Bank:{b_row['Bank_ID']}({b_amt:,.0f}) 有{len(candidates)}筆候選，關鍵字無法分辨 → LLM...")
                best_id = llm_pick_best_candidate(b_row, candidates, mode)
                if best_id:
                    a_sel = candidates[candidates['Acc_ID'] == best_id].iloc[0]
                    history.append({'Type': f'{label} 1對1', 'MatchReason': 'LLM確認',
                                    'Bank_Data': b_row.to_frame().T, 'Acc_Data': a_sel.to_frame().T})
                    rem_a = rem_a[rem_a['Acc_ID'] != best_id]
                    rem_b = rem_b.drop(b_idx)
    # ── Step 2: 1 Bank → 2 Acc ──
    print(f"\n⏳ [Step 2] 1 Bank → 2 Acc... (庫存: 銀行{len(rem_b)} / 會計{len(rem_a)})")
    matched_b, matched_a = [], []
    for (a1i, a1), (a2i, a2) in itertools.combinations(rem_a.iterrows(), 2):
        if a1['Acc_ID'] in matched_a or a2['Acc_ID'] in matched_a: continue
        s = abs(a1['業務金額']) + abs(a2['業務金額'])
        for b_idx, b_row in rem_b[rem_b[bank_amt_col] == s].iterrows():
            if b_row['Bank_ID'] in matched_b: continue
            is_v, rs = evaluate_combination_with_llm(pd.DataFrame([b_row]), pd.DataFrame([a1, a2]), "1對2", mode)
            if is_v:
                print(f"  ✅ 1:2 Bank:{b_row['Bank_ID']} ↔ Acc:{a1['Acc_ID']}+{a2['Acc_ID']}")
                history.append(...)
                matched_b.append(b_row['Bank_ID']); matched_a.extend([a1['Acc_ID'], a2['Acc_ID']])
            else:
                print(f"  🟡 1:2 數學吻合但LLM不確定 → 待確認 Bank:{b_row['Bank_ID']}")
                pending.append({
                    'Mode': mode, 'Type': '1對2 待確認',
                    'Bank_Data': b_row.to_frame().T,
                    'Acc_Data': pd.concat([a1.to_frame().T, a2.to_frame().T]),
                    'Reason': rs
                })


    rem_b = rem_b[~rem_b['Bank_ID'].isin(matched_b)]
    rem_a = rem_a[~rem_a['Acc_ID'].isin(matched_a)]
    # ── Step 3: N Bank → 1 Acc (新增！修正 50M+9.1M+16.1M 的情況) ──
    print(f"\n⏳ [Step 3] N Bank → 1 Acc... (庫存: 銀行{len(rem_b)} / 會計{len(rem_a)})")
    matched_b, matched_a = [], []
    for a_idx, a_row in rem_a.iterrows():
        if a_row['Acc_ID'] in matched_a: continue
        a_amt = abs(a_row['業務金額'])
        if a_amt == 0: continue
        pool = rem_b[(~rem_b['Bank_ID'].isin(matched_b)) & (rem_b[bank_amt_col] <= a_amt)]
        found = False
        for r in range(2, min(5, len(pool) + 1)):
            if found: break
            for combo in itertools.combinations(pool.iterrows(), r):
                rows = [c[1] for c in combo]
                if sum(row[bank_amt_col] for row in rows) != a_amt: continue
                combo_df = pd.DataFrame(rows)
                combo_ids = combo_df['Bank_ID'].tolist()
                print(f"  💡 N:1 候選 Acc:{a_row['Acc_ID']}({a_amt:,.0f}) ← Bank:{combo_ids} → LLM驗證...")
                is_v, rs = evaluate_combination_with_llm(combo_df, a_row.to_frame().T, f"{r}對1", mode)
                if is_v:
                    print(f"  ✅ N:1 Bank:{combo_ids} ↔ Acc:{a_row['Acc_ID']}")
                    history.append(...); matched_b.extend(combo_ids); matched_a.append(a_row['Acc_ID'])
                    found = True; break
                else:
                    print(f"  🟡 N:1 數學吻合但LLM不確定 → 待確認 Bank:{combo_ids}")
                    pending.append({
                        'Mode': mode, 'Type': f'{r}對1 待確認',
                        'Bank_Data': combo_df,
                        'Acc_Data': a_row.to_frame().T,
                        'Reason': rs
                    })


    rem_b = rem_b[~rem_b['Bank_ID'].isin(matched_b)]
    rem_a = rem_a[~rem_a['Acc_ID'].isin(matched_a)]
    # ── Step 4: 同金額 N對N 批次 ──
    print(f"\n⏳ [Step 4] N對N 同金額批次... (庫存: 銀行{len(rem_b)} / 會計{len(rem_a)})")
    for b_amt, _ in rem_b.groupby(bank_amt_col):
        if rem_b.empty or rem_a.empty or b_amt == 0: break
        b_grp = rem_b[rem_b[bank_amt_col] == b_amt]
        a_grp = rem_a[rem_a['業務金額'].abs() == b_amt]
        if len(b_grp) == 0 or len(a_grp) == 0 or len(b_grp) != len(a_grp): continue
        print(f"  ✅ {len(b_grp)}對{len(a_grp)} 批次 金額={b_amt:,.0f}")
        history.append({'Type': f'{label} {len(b_grp)}對{len(a_grp)}批次', 'Bank_Data': b_grp, 'Acc_Data': a_grp})
        rem_b = rem_b[~rem_b['Bank_ID'].isin(b_grp['Bank_ID'])]
        rem_a = rem_a[~rem_a['Acc_ID'].isin(a_grp['Acc_ID'])]
    print(f"\n🎉 [{label}] 完成，已核銷 {len(history)} 筆，剩餘銀行{len(rem_b)}/會計{len(rem_a)}")
    return history, rem_b, rem_a





In [ ]:
# 按日期逐日對帳
import numpy as np
def get_date_str(df, date_col):
    return pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y/%m/%d')
bank_dates = sorted(set(
    get_date_str(bank_in, '交易日期').dropna().tolist() +
    get_date_str(bank_out, '交易日期').dropna().tolist()
))
all_inc_hist, all_exp_hist = [], []
all_inc_rem_b, all_exp_rem_b = [], []
all_inc_rem_a, all_exp_rem_a = [], []

all_pending_combos = []  # 跨天共用
for date in bank_dates:
    b_in_day  = bank_in[get_date_str(bank_in,   '交易日期') == date]
    b_out_day = bank_out[get_date_str(bank_out, '交易日期') == date]
    a_in_day  = acc_in[acc_in['業務日期_格式化']  == date]
    a_out_day = acc_out[acc_out['業務日期_格式化'] == date]
    print(f'\n===== 📅 {date} ===== 銀行收{len(b_in_day)}/支{len(b_out_day)} | 會計收{len(a_in_day)}/支{len(a_out_day)}')
    h_i, r_bi, r_ai = reconcile_engine(b_in_day,  a_in_day,  'Income',  all_pending_combos)
    h_e, r_be, r_ae = reconcile_engine(b_out_day, a_out_day, 'Expense', all_pending_combos)
    all_inc_hist += h_i;  all_exp_hist += h_e
    all_inc_rem_b.append(r_bi); all_exp_rem_b.append(r_be)
    all_inc_rem_a.append(r_ai); all_exp_rem_a.append(r_ae)
inc_hist  = all_inc_hist
exp_hist  = all_exp_hist
inc_rem_b = pd.concat(all_inc_rem_b) if all_inc_rem_b else pd.DataFrame()
exp_rem_b = pd.concat(all_exp_rem_b) if all_exp_rem_b else pd.DataFrame()
inc_rem_a = pd.concat(all_inc_rem_a) if all_inc_rem_a else pd.DataFrame()
exp_rem_a = pd.concat(all_exp_rem_a) if all_exp_rem_a else pd.DataFrame()

In [ ]:
# ===== 階段二：跨日對帳（銀行從最新往前查會計剩餘）=====
print('\n\n========== 🔀 跨日對帳（階段二） ==========')

def cross_day_reconcile(rem_b, rem_a, bank_date_col, acc_date_col, mode, day_window=7):
    history = []
    rem_b, rem_a = rem_b.copy(), rem_a.copy()
    amt_b = '存入金額' if mode == 'Income' else '支出金額'
    label = '收入' if mode == 'Income' else '支出'
    rem_b['_bdate'] = pd.to_datetime(rem_b[bank_date_col], errors='coerce')
    rem_a['_adate'] = pd.to_datetime(rem_a[acc_date_col],  errors='coerce')
    for b_idx, b_row in rem_b.sort_values('_bdate', ascending=False).iterrows():
        b_amt, b_date = b_row[amt_b], b_row['_bdate']
        # ✅ 修正：日期差 ≤ 7天，方向不限
        cands = rem_a[
            (rem_a['業務金額'].abs() == b_amt) &
            ((rem_a['_adate'] - b_date).abs().dt.days <= day_window)
        ].copy()
        if cands.empty: continue
        cands['_diff'] = (cands['_adate'] - b_date).abs().dt.days
        cands = cands.sort_values('_diff')
        if len(cands) == 1:
            a_row = cands.iloc[0]
            print(f"  🔀 跨日[{label}] Bank:{b_row['Bank_ID']}({b_date.date()}) ↔ Acc:{a_row['Acc_ID']}({a_row['_adate'].date()}) 差{cands.iloc[0]['_diff']}天")
            history.append({'Type': f'跨日{label}', 'Bank_Data': b_row.to_frame().T, 'Acc_Data': a_row.to_frame().T})
            rem_a = rem_a.drop(cands.index[0]); rem_b = rem_b.drop(b_idx)
        else:
            best_id = llm_pick_best_candidate(b_row, cands, mode)
            if best_id:
                a_sel = cands[cands['Acc_ID'] == best_id].iloc[0]
                print(f"  🔀 跨日[{label}][LLM] Bank:{b_row['Bank_ID']} ↔ Acc:{best_id}")
                history.append({'Type': f'跨日{label}(LLM)', 'Bank_Data': b_row.to_frame().T, 'Acc_Data': a_sel.to_frame().T})
                rem_a = rem_a[rem_a['Acc_ID'] != best_id]; rem_b = rem_b.drop(b_idx)
    rem_b = rem_b.drop(columns=['_bdate'], errors='ignore')
    rem_a = rem_a.drop(columns=['_adate'], errors='ignore')
    return history, rem_b, rem_a

def final_sweep_cross_day(rem_b, rem_a, bank_date_col, acc_date_col, mode, day_window=7, pending=None):
    if pending is None: pending = []
    history = []
    rem_b = rem_b.copy()
    rem_a = rem_a.copy()
    amt_b = '存入金額' if mode == 'Income' else '支出金額'
    label = '收入' if mode == 'Income' else '支出'
    rem_b['_bdate'] = pd.to_datetime(rem_b[bank_date_col], errors='coerce')
    rem_a['_adate'] = pd.to_datetime(rem_a[acc_date_col],  errors='coerce')
    matched_b, matched_a = [], []

    for b_idx, b_row in rem_b.iterrows():
        if b_row['Bank_ID'] in matched_b: continue
        b_amt  = b_row[amt_b]
        b_date = b_row['_bdate']

        pool = rem_a[
            (~rem_a['Acc_ID'].isin(matched_a)) &
            ((rem_a['_adate'] - b_date).abs().dt.days <= day_window)
        ]

        found = False
        for r in range(2, min(4, len(pool) + 1)):
            if found: break
            for combo in itertools.combinations(pool.iterrows(), r):
                rows = [c[1] for c in combo]
                if sum(abs(row['業務金額']) for row in rows) != b_amt: continue
                combo_df = pd.DataFrame(rows)
                combo_ids = combo_df['Acc_ID'].tolist()

                # LLM 驗證
                is_v, rs = evaluate_combination_with_llm(b_row.to_frame().T, combo_df, f"跨日1對{r}", mode)
                if is_v:
                    print(f"  ✅ 跨日1→{r} [{label}] Bank:{b_row['Bank_ID']}({b_amt:,.0f}) ← Acc:{combo_ids} 核銷")
                    history.append({
                        'Type': f'跨日{label} 1對{r}',
                        'Bank_Data': b_row.to_frame().T,
                        'Acc_Data': combo_df
                    })
                    matched_b.append(b_row['Bank_ID'])
                    matched_a.extend(combo_ids)
                    found = True; break
                else:
                    print(f"  🟡 跨日1→{r} [{label}] Bank:{b_row['Bank_ID']} 數學吻合但LLM不確定 → 待確認")
                    pending.append({
                        'Mode': mode, 'Type': f'跨日{label} 1對{r} 待確認',
                        'Bank_Data': b_row.to_frame().T,
                        'Acc_Data': combo_df,
                        'Reason': rs
                    })

    rem_b = rem_b[~rem_b['Bank_ID'].isin(matched_b)].drop(columns=['_bdate'], errors='ignore')
    rem_a = rem_a[~rem_a['Acc_ID'].isin(matched_a)].drop(columns=['_adate'], errors='ignore')
    return history, rem_b, rem_a


In [ ]:
cross_inc_hist, inc_rem_b, inc_rem_a = cross_day_reconcile(inc_rem_b, inc_rem_a, '交易日期', '業務日期_格式化', 'Income')
cross_exp_hist, exp_rem_b, exp_rem_a = cross_day_reconcile(exp_rem_b, exp_rem_a, '交易日期', '業務日期_格式化', 'Expense')


inc_hist += cross_inc_hist
exp_hist += cross_exp_hist


# ── 在跨日配對後接著執行 ──
sweep_inc_hist, inc_rem_b, inc_rem_a = final_sweep_cross_day(inc_rem_b, inc_rem_a, '交易日期', '業務日期_格式化', 'Income',pending=all_pending_combos)
sweep_exp_hist, exp_rem_b, exp_rem_a = final_sweep_cross_day(exp_rem_b, exp_rem_a, '交易日期', '業務日期_格式化', 'Expense',pending=all_pending_combos)
inc_hist += sweep_inc_hist
exp_hist += sweep_exp_hist
print(f'\n{"="*50}')
print(f'最終剩餘：收入(銀行{len(inc_rem_b)}/會計{len(inc_rem_a)}) | 支出(銀行{len(exp_rem_b)}/會計{len(exp_rem_a)})')
print(f'\n付款剩餘明細：')
if not exp_rem_b.empty:
    print(exp_rem_b[['Bank_ID','交易日期','附言','支出金額']].to_string())
print(f"\n🟡 組合配對待確認：{len(all_pending_combos)} 筆")
for p in all_pending_combos:
    print(f"  [{p['Type']}] {p['Reason']}")
    print(f"  銀行：\n{p['Bank_Data'][['Bank_ID','支出金額' if p['Mode']=='Expense' else '存入金額','附言']].to_string()}")
    print(f"  會計：\n{p['Acc_Data'][['Acc_ID','業務金額','描述.1' if '描述.1' in p['Acc_Data'].columns else '描述']].to_string()}")
    print()

## 結果產出

In [ ]:
# ==========================================
# 掃底模組與 Excel 報表產出
# ==========================================

# --- 1. 掃底模組：尋找金額相符的疑似配對 (Soft Match) ---
def find_soft_matches(rem_b, rem_a, mode):
    soft_matches = []
    amt_col_b = '存入金額' if mode == 'Income' else '支出金額'
    for b_idx, b_row in rem_b.iterrows():
        b_amt = b_row[amt_col_b]
        if b_amt == 0: continue
        candidates = rem_a[rem_a['業務金額'].abs() == b_amt]
        if not candidates.empty:
            soft_matches.append({'Mode': mode, 'Bank_Data': b_row, 'Candidates': candidates})
    return soft_matches

# 執行掃底
inc_soft_matches = find_soft_matches(inc_rem_b, inc_rem_a, 'Income')
exp_soft_matches = find_soft_matches(exp_rem_b, exp_rem_a, 'Expense')
all_soft_matches = inc_soft_matches + exp_soft_matches


# --- 2. Excel 報表產出：以銀行對帳單為主體，在附言右側新增「未入帳」與「待確認」欄 ---
def build_excel_report(bank_path, inc_rem_b, exp_rem_b, soft_matches, output_path,
                       bank_in, bank_out, acc_in, acc_out, pending_combos=None):

    import openpyxl
    from openpyxl import load_workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
    from copy import copy

    soft_ids = set(m['Bank_Data']['Bank_ID'] for m in soft_matches)
    
    # ✅ 新增：pending combo 的銀行也標為「待確認」
    if pending_combos:
        for p in pending_combos:
            for bid in p['Bank_Data']['Bank_ID'].values:
                soft_ids.add(str(bid))
    all_rem_b = pd.concat([inc_rem_b, exp_rem_b])
    hard_rem_ids = set(all_rem_b[~all_rem_b['Bank_ID'].isin(soft_ids)]['Bank_ID'].astype(str))
    pending_ids  = soft_ids

    wb = load_workbook(bank_path)
    # 找含 target_date（或 '0409'）關鍵字的工作表
    target_sheet = None
    for name in wb.sheetnames:
        if '0409' in name or '09' in name:
            target_sheet = name
            break
    ws = wb[target_sheet] if target_sheet else wb.active
    print(f'使用工作表：{ws.title}')

    # 找「附言」欄位
    header_row = None
    memo_col = None
    for row in ws.iter_rows():
        for cell in row:
            if str(cell.value).strip() == '附言':
                header_row = cell.row
                memo_col = cell.column
                break
        if header_row:
            break

    if memo_col is None:
        insert_col = ws.max_column + 1
    else:
        insert_col = memo_col + 1
        ws.insert_cols(insert_col, 2)

    center = Alignment(horizontal='center', vertical='center')

    # 複製附言欄標題的樣式（框線＋背景色）來作為新欄標題樣式
    ref_header = ws.cell(row=header_row, column=memo_col)

    for offset, col_name in enumerate(['未入帳', '待確認']):
        c = ws.cell(row=header_row, column=insert_col + offset, value=col_name)
        c.font      = copy(ref_header.font)
        c.fill      = copy(ref_header.fill)
        c.border    = copy(ref_header.border)
        c.alignment = copy(ref_header.alignment)

    # 資料列：複製附言欄的框線與背景色，並依狀況印 V
    skip = 5
    data_start_row = header_row + 1
    # 先把所有資料列複製樣式（不標記）
    for excel_row in range(data_start_row, ws.max_row + 1):
        ref_data = ws.cell(row=excel_row, column=memo_col)
        for col_offset in [0, 1]:
            c = ws.cell(row=excel_row, column=insert_col + col_offset)
            c.fill      = copy(ref_data.fill)
            c.border    = copy(ref_data.border)
            c.alignment = center
            c.font      = copy(ref_data.font)
    # 直接用 Bank_ID 算出對應的 Excel 列號，標記 V
    for bank_id in hard_rem_ids:
        excel_row = data_start_row + int(bank_id)
        if excel_row <= ws.max_row:
            ws.cell(row=excel_row, column=insert_col).value = 'V'
            ws.cell(row=excel_row, column=insert_col).font  = Font(bold=True, color='FF0000')
    for bank_id in pending_ids:
        excel_row = data_start_row + int(bank_id)
        if excel_row <= ws.max_row:
            ws.cell(row=excel_row, column=insert_col + 1).value = 'V'
            ws.cell(row=excel_row, column=insert_col + 1).font  = Font(bold=True, color='FF8C00')

    ws.column_dimensions[openpyxl.utils.get_column_letter(insert_col)].width   = 10
    ws.column_dimensions[openpyxl.utils.get_column_letter(insert_col + 1)].width = 10
        # --- 在最後一列後面加入淨額彙總 ---
    # --- 按日期分天彙總 ---
    summary_row = ws.max_row + 2
    bold = Font(bold=True)
    header_fill = PatternFill('solid', fgColor='D9E1F2')
    center = Alignment(horizontal='center')
    # 取得所有涉及日期
    bank_df = pd.concat([bank_in, bank_out])
    acc_df  = pd.concat([acc_in,  acc_out])
    bank_df['_date'] = pd.to_datetime(bank_df['交易日期'], errors='coerce').dt.strftime('%Y/%m/%d')
    acc_df['_date']  = acc_df['業務日期_格式化'] if '業務日期_格式化' in acc_df.columns else pd.to_datetime(acc_df['業務日期'], errors='coerce').dt.strftime('%Y/%m/%d')
    all_dates = sorted(set(bank_df['_date'].dropna()) | set(acc_df['_date'].dropna()))
    # 標題列
    col_start = 1
    headers_bank = ['交易日期', '銀行收入', '銀行支出', '銀行變動數']
    headers_acc  = ['交易日期', '會計收入', '會計支出', '會計變動數']
    # 大標題
    c = ws.cell(row=summary_row, column=col_start, value='銀行對賬單')
    c.font = Font(bold=True, size=12); c.fill = header_fill
    c = ws.cell(row=summary_row, column=col_start + 4, value='會計賬務')
    c.font = Font(bold=True, size=12); c.fill = PatternFill('solid', fgColor='E2EFDA')
    # 欄位標題
    for i, h in enumerate(headers_bank):
        c = ws.cell(row=summary_row+1, column=col_start+i, value=h)
        c.font = bold; c.fill = header_fill; c.alignment = center
    for i, h in enumerate(headers_acc):
        c = ws.cell(row=summary_row+1, column=col_start+4+i, value=h)
        c.font = bold; c.fill = PatternFill('solid', fgColor='E2EFDA'); c.alignment = center
    # 每一天的資料
    for r, date in enumerate(all_dates):
        row = summary_row + 2 + r
        b_in  = bank_df[(bank_df['_date']==date) & (bank_df['存入金額']>0)]['存入金額'].sum() if '存入金額' in bank_df else 0
        b_out = bank_df[(bank_df['_date']==date) & (bank_df['支出金額']>0)]['支出金額'].sum() if '支出金額' in bank_df else 0
        a_in  = acc_df[(acc_df['_date']==date)  & (acc_df['業務金額']>0)]['業務金額'].sum()
        a_out = abs(acc_df[(acc_df['_date']==date) & (acc_df['業務金額']<0)]['業務金額'].sum())
        for col, val in zip(
            [col_start, col_start+1, col_start+2, col_start+3,
             col_start+4, col_start+5, col_start+6, col_start+7],
            [date, b_in, b_out, b_in-b_out, date, a_in, a_out, abs(a_in-a_out)]
        ):
            c = ws.cell(row=row, column=col, value=val)
            c.alignment = center
            if isinstance(val, float) or (isinstance(val, (int,)) and col != col_start and col != col_start+4):
                c.number_format = '#,##0'

    wb.save(output_path)
    print(f'\n💾 Excel 報表已產出：{output_path}')

# 執行產出
# 執行產出
build_excel_report(
    bank_path, inc_rem_b, exp_rem_b,
    all_soft_matches, output_xlsx,
    bank_in, bank_out, acc_in, acc_out,
    pending_combos=all_pending_combos  # ✅ 新增
)


In [ ]:
def build_html_report_with_soft_match(all_histories, all_rems, soft_matches, date, output_path, pending_combos=None):
    style = """
    <style>
        body { font-family: 'Inter', system-ui, sans-serif; background: #f8fafc; color: #1e293b; padding: 40px; }
        .container { max-width: 1400px; margin: auto; }
        h1 { color: #0f172a; border-bottom: 3px solid #3b82f6; padding-bottom: 10px; }
        .summary-box { display: grid; grid-template-columns: repeat(3, 1fr); gap: 20px; margin-bottom: 30px; }
        .stat-card { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 6px rgb(0 0 0 / 0.05); }
        .card { background: white; border-radius: 16px; padding: 25px; margin-bottom: 40px; box-shadow: 0 10px 15px -3px rgb(0 0 0 / 0.1); }
        .tag { display: inline-block; padding: 4px 12px; border-radius: 999px; font-size: 12px; font-weight: bold; margin-bottom: 15px; }
        .tag-inc { background: #dcfce7; color: #166534; } 
        .tag-exp { background: #fee2e2; color: #991b1b; }
        .tag-warn { background: #fef08a; color: #854d0e; } 
        .table-section { margin-top: 15px; overflow-x: auto; }
        .side-title { font-size: 13px; text-transform: uppercase; font-weight: bold; color: #64748b; margin-bottom: 10px; border-left: 4px solid #3b82f6; padding-left: 10px; }
        table { width: 100%; border-collapse: collapse; font-size: 12px; margin-bottom: 10px; text-align: right; }
        th { background: #f8fafc; padding: 10px; border-bottom: 2px solid #e2e8f0; text-align: right; }
        td { padding: 10px; border-bottom: 1px solid #f1f5f9; }
    </style>
    """
    
    total_matched = len(all_histories['Income']) + len(all_histories['Expense'])
    total_pending_all = len(soft_matches) + len(pending_combos or [])
    
    # 計算銀行端真正未被匹配的數量
    rem_b_ids_in_soft = [m['Bank_Data']['Bank_ID'] for m in soft_matches]
    total_hard_rem = len(all_rems['Income_B'][~all_rems['Income_B']['Bank_ID'].isin(rem_b_ids_in_soft)]) + \
                     len(all_rems['Expense_B'][~all_rems['Expense_B']['Bank_ID'].isin(rem_b_ids_in_soft)])
    
    html = f"<div class='container'><h1>收支核銷報告 ({date})</h1>"
    
    # === 頂部三大統計區塊 ===
    html += f"<div class='summary-box'>"
    html += f"<div class='stat-card' style='border-top: 4px solid #22c55e;'><h3>🟢 總結成功核銷</h3><div style='font-size:32px; font-weight:bold; color:#22c55e;'>{total_matched} 筆</div></div>"
    html += f"<div class='stat-card' style='border-top: 4px solid #eab308;'><h3>🟡 全部待確認</h3><div style='font-size:32px; font-weight:bold; color:#eab308;'>{total_pending_all} 筆</div></div>"
    html += f"<div class='stat-card' style='border-top: 4px solid #ef4444;'><h3>🔴 總計剩餘未入帳 (銀行端)</h3><div style='font-size:32px; font-weight:bold; color:#ef4444;'>{total_hard_rem} 筆</div></div>"
    html += "</div>"
    
    # === 🟡 系統猜測可能配對區塊 (改為詳細資料表呈現) ===
    # ✅ 合併成一段
    if soft_matches or pending_combos:
        total_p = len(soft_matches) + len(pending_combos or [])
        html += f"<h2>🟡 全部待確認（共 {total_p} 筆）</h2>"
        html += "<p style='color: #64748b; font-size: 14px;'>以下為金額吻合但需人工確認的紀錄，包含掃底配對及組合配對。</p>"
        # 先放 soft_matches（1:1 金額相符）
        for idx, match in enumerate(soft_matches):
            mode_tw = '收款' if match['Mode'] == 'Income' else '付款'
            b_data = match['Bank_Data']
            html += f"<div class='card' style='border-left: 5px solid #eab308; background-color: #fefce8; padding: 20px;'>"
            html += f"<div class='tag tag-warn'>待確認 {mode_tw}（1對1金額相符）</div>"
            html += "<div class='table-section'><div class='side-title'>【銀行端】</div>"
            html += b_data.to_frame().T.to_html(index=False)
            html += "</div>"
            html += f"<div class='table-section' style='margin-top:20px;'><div class='side-title'>👇 會計端候選（{len(match['Candidates'])} 筆）</div>"
            html += match['Candidates'].to_html(index=False)
            html += "</div></div>"
        # 再放 pending_combos（組合配對 LLM 存疑）
        for p in (pending_combos or []):
            html += f"<div class='card' style='border-left:5px solid #eab308;background:#fefce8;padding:20px;'>"
            html += f"<div class='tag tag-warn'>{p['Type']}</div>"
            html += f"<div style='color:#92400e;margin-bottom:10px;'>LLM意見：{p['Reason']}</div>"
            html += "<div class='table-section'><div class='side-title'>【銀行端】</div>"
            html += p['Bank_Data'].to_html(index=False)
            html += "</div><div class='table-section' style='margin-top:15px;'><div class='side-title'>【會計端】</div>"
            html += p['Acc_Data'].to_html(index=False)
            html += "</div></div>"

    # === 🔴 剩餘未入帳清單 ===
    html += "<h2 style='margin-top:40px'>🔴 剩餘未入帳之銀行賬務清單 (待盤點)</h2>"
    p_map = {'Income': '尚未入帳之【收款】', 'Expense': '尚未入帳之【付款】'}
    for m in ['Income', 'Expense']:
        df_rem = all_rems[f'{m}_B']
        df_rem_pure = df_rem[~df_rem['Bank_ID'].isin(rem_b_ids_in_soft)]
        
        if not df_rem_pure.empty:
            html += f"<div class='card' style='border-top: 5px solid {'#22c55e' if m=='Income' else '#ef4444'}'>"
            html += f"<div class='tag {'tag-inc' if m=='Income' else 'tag-exp'}'>{p_map[m]}</div>"
            html += df_rem_pure[['Bank_ID', '交易日期', '交易時間', '附言', '摘要', '存入金額', '支出金額']].to_html(index=False)
            html += "</div>"
            
    # === 🟢 詳盡對帳軌跡 ===
    html += "<h2 style='margin-top:60px'>🟢 詳盡對帳軌跡 (已核銷)</h2>"
    all_matched = all_histories['Income'] + all_histories['Expense']
    for idx, item in enumerate(all_matched):
        is_inc = '收入' in item['Type']
        reason = item.get('MatchReason', '—')
        html += f"<div class='card'>"
        html += f"<div class='tag {'tag-inc' if is_inc else 'tag-exp'}'>{item['Type']}</div>"
        html += f"<div style='font-size:12px;color:#64748b;margin-bottom:10px;'>⚙️ 配對依據：{reason}</div>"
        
        html += "<div class='table-section'><div class='side-title'>銀行端原始紀錄</div>"
        html += item['Bank_Data'].to_html(index=False)
        html += "</div>"
        
        html += "<div class='table-section'><div class='side-title'>會計端原始紀錄</div>"
        html += item['Acc_Data'].to_html(index=False)
        html += "</div></div>"
    
    html += "</div>"
    with open(output_path, 'w', encoding='utf-8') as f: 
        f.write(f"<html><head><meta charset='utf-8'>{style}</head><body>{html}</body></html>")
# 過濾掉 history 裡的 Ellipsis
inc_hist = [item for item in inc_hist if isinstance(item, dict)]
exp_hist = [item for item in exp_hist if isinstance(item, dict)]
print(f"清理後：inc_hist {len(inc_hist)} 筆 / exp_hist {len(exp_hist)} 筆")

# HTML 報告
output_html = os.path.join(data_dir, base_name + '.html')
build_html_report_with_soft_match(
    {'Income': inc_hist, 'Expense': exp_hist},
    {'Income_B': inc_rem_b, 'Expense_B': exp_rem_b},
    all_soft_matches,
    target_date,
    output_html,
    pending_combos=all_pending_combos  # ✅ 新增
)